# GLS map-making of a `litebird_sim` simulation with pysanepic

This notebook simulates one day of LiteBIRD observations with `litebird_sim`
(two LFT 140 GHz detectors, realistic scanning strategy, rotating HWP, CMB sky,
1/f noise with the IMO parameters), then makes maps with:

1. the `litebird_sim` **binner** (no noise model),
2. the `litebird_sim` **destriper**,
3. **pysanepic** GLS, which uses the full 1/f noise spectrum.

All maps are in Ecliptic coordinates and are compared with the binned map of
the noiseless signal, so the residuals contain only noise.

Requirements: `litebird_sim` and `pysanepic` installed in the same environment.
The simulation takes about a minute on a laptop.

In [ ]:
import time

import astropy.time
import healpy as hp
import litebird_sim as lbs
import matplotlib.pyplot as plt
import numpy as np

from pysanepic import GLS, one_over_f_weights

## Parameters

In [ ]:
telescope, channel = "LFT", "L4-140"
detlist = ["000_001_017_QB_140_T", "000_001_017_QB_140_B"]
duration_days = 1.0
nside = 64

# pysanepic applies N^-1 independently on chunks of data: 1 hour here.
chunk_s = 3600.0
# Destriper baseline length (the lbs default of 100 samples is far too short here)
baseline_s = 600.0

## Simulation with `litebird_sim`

In [ ]:
imo = lbs.Imo(flatfile_location=lbs.PTEP_IMO_LOCATION)
sim = lbs.Simulation(
    base_path="lbs_example_output",
    imo=imo,
    start_time=astropy.time.Time("2030-01-01T00:00:00"),
    duration_s=duration_days * 86400,
    random_seed=12345,
)
sim.set_instrument(
    lbs.InstrumentInfo.from_imo(imo, f"/releases/vPTEP/satellite/{telescope}/instrument_info")
)
dets = [
    lbs.DetectorInfo.from_imo(
        url=f"/releases/vPTEP/satellite/{telescope}/{channel}/{d}/detector_info", imo=imo
    )
    for d in detlist
]
sim.set_scanning_strategy(imo_url="/releases/vPTEP/satellite/scanning_parameters/")
sim.create_observations(detectors=dets)
sim.set_hwp(lbs.IdealHWP(sim.instrument.hwp_rpm * 2 * np.pi / 60))
sim.prepare_pointings()

ch = lbs.FreqChannelInfo.from_imo(
    url=f"/releases/vPTEP/satellite/{telescope}/{channel}/channel_info", imo=imo
)
sky = sim.get_sky(
    parameters=lbs.SkyGenerationParams(
        make_cmb=True, make_fg=False, seed_cmb=1, apply_beam=True,
        units="K_CMB", output_type="map", nside=nside,
    ),
    channels=[ch],
    store_in_observation=True,
)
sim.fill_tods()

obs = sim.observations[0]
signal = obs.tod.copy()  # kept to build the noiseless reference map
sim.add_noise(noise_type="one_over_f")

fs = obs.sampling_rate_hz
print(f"{obs.n_detectors} detectors x {obs.n_samples} samples at {fs} Hz")
print(f"fknee = {obs.fknee_mhz} mHz, alpha = {obs.alpha}, NET = {obs.net_ukrts} uK sqrt(s)")

## From `litebird_sim` observations to pysanepic arrays

pysanepic works on plain arrays with one row per *chunk* (one detector × one
stretch of time). For each sample we need:

* the pixel index, from the (θ, φ) pointing;
* the polarization angle. With an ideal HWP, `litebird_sim` uses
  ψ = ψ_pointing + 2·φ_HWP − ψ_det (see `_get_pol_angle` in `litebird_sim`);
* the TOD.

The noise weights are the inverse of the `litebird_sim` 1/f PSD,
P(f) = σ² (f^α + f_knee^α) / (f^α + f_min^α), with σ = NET·√f_s.

*A proper `litebird_sim` interface in pysanepic will replace this cell.*

In [ ]:
pointings, hwp_angle = obs.get_pointings("all")
pix = hp.ang2pix(nside, pointings[..., 0], pointings[..., 1])
psi = pointings[..., 2] + 2 * hwp_angle - obs.pol_angle_rad[:, None]

ns = int(chunk_s * fs)
nchunk = obs.n_samples // ns
used = nchunk * ns
print(f"{nchunk} chunks of {ns} samples per detector, {obs.n_samples - used} samples dropped")


def chunks(a):
    return a[:, :used].reshape(-1, ns)


weights = np.concatenate([
    np.broadcast_to(
        one_over_f_weights(
            ns, fs,
            sigma=obs.net_ukrts[i] * np.sqrt(fs) / 1e6,
            fknee_hz=obs.fknee_mhz[i] / 1e3,
            alpha=obs.alpha[i],
            fmin_hz=obs.fmin_hz[i],
        ),
        (nchunk, ns // 2 + 1),
    )
    for i in range(obs.n_detectors)
])

## GLS map-making

In [ ]:
t0 = time.time()
gls = GLS(chunks(pix), chunks(psi), weights)
t1 = time.time()
m, info = gls.solve(chunks(obs.tod), tol=1e-12)
t2 = time.time()
print(f"setup {t1 - t0:.1f} s, solve {t2 - t1:.1f} s, {info['iterations']} iterations")
gls_map = gls.to_healpix(m, nside)

plt.semilogy(info["residuals"])
plt.xlabel("iteration")
plt.ylabel(r"$|r|^2 / |b|^2$")
plt.title("PCG convergence");

## Binner and destriper from `litebird_sim`, and the noiseless reference

In [ ]:
ecl = lbs.CoordinateSystem.Ecliptic
binned = sim.make_binned_map(nside=nside, output_coordinate_system=ecl).binned_map
destriped = sim.make_destriped_map(
    nside=nside,
    params=lbs.DestriperParameters(
        output_coordinate_system=ecl,
        samples_per_baseline=int(baseline_s * fs),
        threshold=1e-12,
        iter_max=500,
    ),
).destriped_map

noisy = obs.tod.copy()
obs.tod[:] = signal
reference = sim.make_binned_map(nside=nside, output_coordinate_system=ecl).binned_map
obs.tod[:] = noisy

## Comparison

Residual = map − noiseless reference. The I monopole is removed, since none of
the map-makers constrains it. The white-noise floor is σ²/hits per pixel,
i.e. what an ideal map-maker would reach if there were no 1/f noise.

In [ ]:
seen = np.isfinite(gls_map[0]) & np.isfinite(gls_map[1])
residuals = {}
for name, mp in [("binned", binned), ("destriped", destriped), ("pysanepic GLS", gls_map)]:
    r = np.where(seen, mp - reference, np.nan)
    r[0] -= np.nanmean(r[0])
    residuals[name] = r

hits = np.bincount(pix.ravel(), minlength=12 * nside**2)
sigma_k = obs.net_ukrts[0] * np.sqrt(fs) / 1e6
print(f"white-noise floor, I: {1e6 * np.sqrt(np.mean(sigma_k**2 / hits[seen])):.2f} uK")
print(f"{'':15s}   rms residual I / Q / U [uK]")
for name, r in residuals.items():
    print(f"{name:15s}   " + " / ".join(f"{1e6 * np.nanstd(x):6.2f}" for x in r))

In [ ]:
plt.figure(figsize=(15, 4))
for i, (name, r) in enumerate(residuals.items()):
    hp.mollview(1e6 * r[0], sub=(1, 3, i + 1), title=f"{name}: I residual", unit="uK",
                min=-100, max=100, cmap="RdBu_r", badcolor="lightgray")

Q and U residuals are essentially the same for all three map-makers: the HWP
modulates polarization at 4·f_HWP ≈ 3 Hz, far above the 20 mHz knee, so
polarization is not affected by 1/f noise. In intensity, the 1/f noise adds
on top of the white-noise floor: GLS gets closest to the floor, the destriper
is close behind (only with long baselines), and the binner is highest. With
one day of data the differences are small; they grow with longer simulations
and higher f_knee.